In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/kalpe/projects/adaptive_rl_anomaly_detection


In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available. PPO will run on CPU.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA version: 13.0


In [3]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [4]:
from src.rl.state import (
    S3StateBuilder,
    STATE_DIM,
    STATE_FEATURE_NAMES,
    TRAFFIC_FEATURE_NAMES,
    MODEL_ORDER,
)

from src.rl.action import (
    ActionProjector,
    ACTION_DIM,
    DETECTOR_NAMES,
)

from src.rl.reward import (
    FINAL_REWARD_SCHEME,
)

from src.rl.environment import (
    AnomalyEnsembleEnv,
)

from src.rl.agent import (
    PPOAgent,
    PPOConfig,
)

from src.rl.trainer import (
    Trainer,
    TrainerConfig,
)

print("STATE_DIM:", STATE_DIM)
print("ACTION_DIM:", ACTION_DIM)
print("Detector order:", DETECTOR_NAMES)
print("Reward:", FINAL_REWARD_SCHEME.name)

STATE_DIM: 14
ACTION_DIM: 4
Detector order: ('isolation_forest', 'lof', 'ocsvm', 'autoencoder')
Reward: R1_balanced


In [5]:
assert STATE_DIM == 14
assert ACTION_DIM == 4

assert DETECTOR_NAMES == (
    "isolation_forest",
    "lof",
    "ocsvm",
    "autoencoder",
)

assert MODEL_ORDER == DETECTOR_NAMES

print("RL experimental design verified.")

RL experimental design verified.


In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PPO device:", DEVICE)

if DEVICE == "cuda":
    print("Using:", torch.cuda.get_device_name(0))

PPO device: cuda
Using: NVIDIA GeForce RTX 5060 Laptop GPU


In [7]:
ppo_config = PPOConfig(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,

    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,

    clip_coef=0.20,
    ent_coef=0.01,
    vf_coef=0.5,

    max_grad_norm=0.5,

    n_epochs=10,
    batch_size=64,
    rollout_steps=2048,

    device=DEVICE,
)

print(ppo_config)

PPOConfig(state_dim=14, action_dim=4, hidden_dim=128, max_action=5.0, lr=0.0003, gamma=0.99, gae_lambda=0.95, clip_coef=0.2, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5, n_epochs=10, batch_size=64, rollout_steps=2048, device='cuda')


In [8]:
agent = PPOAgent(ppo_config)

print("PPO agent created.")
print("Device:", agent.device)

PPO agent created.
Device: cuda


In [9]:
print(agent.actor)
print()
print(agent.critic)

Actor(
  (net): Sequential(
    (0): Linear(in_features=14, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)

Critic(
  (net): Sequential(
    (0): Linear(in_features=14, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

In [11]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "datasets"
    / "processed"
    / "cicids2017_processed.parquet"
)

STATE_S3_PATH = (
    PROJECT_ROOT
    / "evaluation_results"
    / "rl_states"
    / "state_s3.npy"
)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_PATH)
print("S3 state:", STATE_S3_PATH)

assert PROCESSED_PATH.exists()
assert STATE_S3_PATH.exists()

Project root: /home/kalpe/projects/adaptive_rl_anomaly_detection
Processed data: /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/datasets/processed/cicids2017_processed.parquet
S3 state: /home/kalpe/projects/adaptive_rl_anomaly_detection/evaluation_results/rl_states/state_s3.npy


In [12]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA: 13.0
Selected device: cuda


In [13]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [14]:
df = pd.read_parquet(PROCESSED_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.shape[1])

assert "Label" in df.columns

Dataset shape: (2520798, 71)
Columns: 71


In [15]:
X = df.drop(columns=["Label"])
y = (df["Label"] != 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_dev, X_val, y_dev, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train,
)

X_dev = X_dev.reset_index(drop=True)
y_dev = y_dev.reset_index(drop=True)

X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Development:", X_dev.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

print("\nValidation labels:")
print(y_val.value_counts().sort_index())

Development: (1814974, 70)
Validation : (201664, 70)
Test       : (504160, 70)

Validation labels:
Label
0    167605
1     34059
Name: count, dtype: int64


In [16]:
state_s3 = np.load(STATE_S3_PATH)

print("S3 shape:", state_s3.shape)
print("S3 dtype:", state_s3.dtype)

S3 shape: (201664, 14)
S3 dtype: float64


In [17]:
assert state_s3.shape[0] == len(y_val)
assert state_s3.shape[1] == 14

print("S3 rows:", state_s3.shape[0])
print("Validation labels:", len(y_val))
print("State dimensions:", state_s3.shape[1])

print("\n[OK] S3 state and validation labels have matching sample counts.")

S3 rows: 201664
Validation labels: 201664
State dimensions: 14

[OK] S3 state and validation labels have matching sample counts.


In [18]:
print("Detector score ranges:")

for i, name in enumerate([
    "Isolation Forest",
    "LOF",
    "OCSVM",
    "Autoencoder",
]):
    print(
        f"{i}: {name:18s} "
        f"min={state_s3[:, i].min():.6f}, "
        f"max={state_s3[:, i].max():.6f}"
    )

assert np.all(state_s3[:, :4] >= 0.0)
assert np.all(state_s3[:, :4] <= 1.0)

print("\n[OK] detector score columns are normalized to [0, 1].")

Detector score ranges:
0: Isolation Forest   min=0.000005, max=1.000000
1: LOF                min=0.000005, max=1.000000
2: OCSVM              min=0.000005, max=1.000000
3: Autoencoder        min=0.000005, max=1.000000

[OK] detector score columns are normalized to [0, 1].


In [20]:
from src.rl.state import (
    STATE_DIM,
    STATE_FEATURE_NAMES,
    TRAFFIC_FEATURE_NAMES,
)

from src.rl.action import (
    ACTION_DIM,
    DETECTOR_NAMES,
    ActionProjector,
)

from src.rl.reward import FINAL_REWARD_SCHEME

from src.rl.environment import AnomalyEnsembleEnv

from src.rl.agent import PPOAgent, PPOConfig

from src.rl.trainer import Trainer, TrainerConfig

In [21]:
assert STATE_DIM == 14
assert ACTION_DIM == 4

assert DETECTOR_NAMES == (
    "isolation_forest",
    "lof",
    "ocsvm",
    "autoencoder",
)

assert FINAL_REWARD_SCHEME.name == "R1_balanced"

print("State dimension :", STATE_DIM)
print("Action dimension:", ACTION_DIM)
print("Detector order  :", DETECTOR_NAMES)
print("Reward scheme   :", FINAL_REWARD_SCHEME.name)

print("\n[OK] RL experimental design verified.")

State dimension : 14
Action dimension: 4
Detector order  : ('isolation_forest', 'lof', 'ocsvm', 'autoencoder')
Reward scheme   : R1_balanced

[OK] RL experimental design verified.


In [22]:
val_env = AnomalyEnsembleEnv(
    states=state_s3.astype(np.float32),
    y_true=y_val.to_numpy(),
    threshold=0.5,
    action_projector=ActionProjector(method="softmax"),
    reward_scheme=FINAL_REWARD_SCHEME,
    shuffle=False,
    seed=SEED,
)

print("Validation environment created.")
print("Samples:", val_env.n_samples)

Validation environment created.
Samples: 201664


In [23]:
import joblib

MODEL_DIR = PROJECT_ROOT / "notebooks" / "trained_models"

if_models = {
    "isolation_forest": joblib.load(MODEL_DIR / "isolation_forest.joblib"),
    "lof": joblib.load(MODEL_DIR / "local_outlier_factor.joblib"),
    "ocsvm": joblib.load(MODEL_DIR / "one_class_svm.joblib"),
    "autoencoder": joblib.load(MODEL_DIR / "autoencoder.joblib"),
}

for name, model in if_models.items():
    print(f"{name}: {type(model)}")

isolation_forest: <class 'src.models.isolation_forest.IsolationForestModel'>
lof: <class 'src.models.local_outlier_factor.LocalOutlierFactorModel'>
ocsvm: <class 'src.models.one_class_svm.OneClassSVMModel'>
autoencoder: <class 'src.models.autoencoder.AutoEncoderModel'>


In [25]:
ae = if_models["autoencoder"]

print("Autoencoder device:", ae._device)
print("Model device:", next(ae._model.parameters()).device)

Autoencoder device: cuda
Model device: cpu


In [26]:
print("X_dev type:", type(X_dev))
print("X_dev shape:", X_dev.shape)

X_dev type: <class 'pandas.DataFrame'>
X_dev shape: (1814974, 70)


In [27]:
ae = if_models["autoencoder"]

ae._device = torch.device(DEVICE)
ae._model = ae._model.to(ae._device)

print("Autoencoder device:", ae._device)
print("Model device:", next(ae._model.parameters()).device)

Autoencoder device: cuda
Model device: cuda:0


In [28]:
dev_scores = {
    "isolation_forest": np.asarray(
        if_models["isolation_forest"].anomaly_score(X_dev)
    ),
    "lof": np.asarray(
        if_models["lof"].anomaly_score(X_dev)
    ),
    "ocsvm": np.asarray(
        if_models["ocsvm"].anomaly_score(X_dev)
    ),
    "autoencoder": np.asarray(
        if_models["autoencoder"].anomaly_score(X_dev)
    ),
}

for name, scores in dev_scores.items():
    print(
        f"{name:18s}: "
        f"shape={scores.shape}, "
        f"min={scores.min():.6f}, "
        f"max={scores.max():.6f}"
    )

isolation_forest  : shape=(1814974,), min=0.319992, max=0.767516
lof               : shape=(1814974,), min=-2.377008, max=418594.157933
ocsvm             : shape=(1814974,), min=-75.389353, max=100.205412
autoencoder       : shape=(1814974,), min=0.000068, max=42782.078125


In [29]:
for name, scores in dev_scores.items():
    assert scores.shape == (len(X_dev),)
    assert np.all(np.isfinite(scores))

assert len(y_dev) == len(X_dev)

print("X_dev :", X_dev.shape)
print("y_dev :", y_dev.shape)

print("\n[OK] All development detector scores align with X_dev/y_dev.")

X_dev : (1814974, 70)
y_dev : (1814974,)

[OK] All development detector scores align with X_dev/y_dev.


In [32]:
val_scores = {
    "isolation_forest": np.asarray(
        if_models["isolation_forest"].anomaly_score(X_val)
    ),
    "lof": np.asarray(
        if_models["lof"].anomaly_score(X_val)
    ),
    "ocsvm": np.asarray(
        if_models["ocsvm"].anomaly_score(X_val)
    ),
    "autoencoder": np.asarray(
        if_models["autoencoder"].anomaly_score(X_val)
    ),
}

for name, scores in val_scores.items():
    print(
        f"{name:18s}: "
        f"shape={scores.shape}, "
        f"min={scores.min():.6f}, "
        f"max={scores.max():.6f}"
    )

isolation_forest  : shape=(201664,), min=0.319992, max=0.744344
lof               : shape=(201664,), min=-2.296478, max=8985.527440
ocsvm             : shape=(201664,), min=-73.646861, max=100.205412
autoencoder       : shape=(201664,), min=0.000068, max=2774.155029


In [33]:
from src.rl.state import S3StateBuilder

state_builder = S3StateBuilder()

# Fit score normalizers using the reference validation scores.
state_builder.fit_normalizers(val_scores)

dev_state_s3 = state_builder.build(
    raw_scores=dev_scores,
    traffic_features=X_dev,
)

print("Development S3 shape:", dev_state_s3.shape)
print("dtype:", dev_state_s3.dtype)

Development S3 shape: (1814974, 14)
dtype: float32


In [34]:
assert dev_state_s3.shape == (len(X_dev), 14)
assert dev_state_s3.dtype == np.float32

assert np.all(np.isfinite(dev_state_s3))

# Detector-score columns must be in [0, 1].
assert np.all(dev_state_s3[:, :4] >= 0.0)
assert np.all(dev_state_s3[:, :4] <= 1.0)

print("[OK] Development S3 state is valid.")
print("Shape:", dev_state_s3.shape)

[OK] Development S3 state is valid.
Shape: (1814974, 14)


In [35]:
train_env = AnomalyEnsembleEnv(
    states=dev_state_s3,
    y_true=y_dev.to_numpy(),
    threshold=0.5,
    action_projector=ActionProjector(method="softmax"),
    reward_scheme=FINAL_REWARD_SCHEME,
    shuffle=True,
    seed=SEED,
)

print("Training environment created.")
print("Training samples:", train_env.n_samples)

Training environment created.
Training samples: 1814974


In [36]:
val_env = AnomalyEnsembleEnv(
    states=state_s3.astype(np.float32),
    y_true=y_val.to_numpy(),
    threshold=0.5,
    action_projector=ActionProjector(method="softmax"),
    reward_scheme=FINAL_REWARD_SCHEME,
    shuffle=False,
    seed=SEED,
)

print("Validation environment created.")
print("Validation samples:", val_env.n_samples)

Validation environment created.
Validation samples: 201664


In [37]:
agent_config = PPOConfig(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    device=DEVICE,
)

print(agent_config)

PPOConfig(state_dim=14, action_dim=4, hidden_dim=128, max_action=5.0, lr=0.0003, gamma=0.99, gae_lambda=0.95, clip_coef=0.2, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5, n_epochs=10, batch_size=64, rollout_steps=2048, device='cuda')


In [38]:
agent = PPOAgent(agent_config)

print("PPO agent created.")
print("Device:", agent.device)

PPO agent created.
Device: cuda


In [40]:
state = train_env.reset()

raw_action, log_prob, value = agent.step(state)

print("State shape:", state.shape)
print("Raw action:", raw_action)
print("Raw action shape:", raw_action.shape)
print("Log probability:", log_prob)
print("Value:", value)

assert state.shape == (STATE_DIM,)
assert raw_action.shape == (ACTION_DIM,)
assert np.all(np.isfinite(raw_action))

print("\n[OK] PPO agent produces a valid raw action.")

State shape: (14,)
Raw action: [ 0.13354842 -0.09479506 -0.71305025  0.76992494]
Raw action shape: (4,)
Log probability: -4.234894752502441
Value: -0.0771128460764885

[OK] PPO agent produces a valid raw action.


In [41]:
result = train_env.step(raw_action)

print("Reward:", result.reward)
print("Done:", result.done)
print("Ensemble score:", result.info["ensemble_score"])

print("Projected weights:", result.info["weights"])
print("Weight sum:", result.info["weights"].sum())

print("True label:", result.info["y_true"])
print("Prediction:", result.info["y_pred"])

assert np.isfinite(result.reward)
assert result.info["weights"].shape == (ACTION_DIM,)
assert np.isclose(result.info["weights"].sum(), 1.0, atol=1e-5)
assert np.all(result.info["weights"] >= 0.0)

print("\n[OK] PPO raw action → projected weights → ensemble → reward works.")

Reward: 1.0
Done: False
Ensemble score: 0.23106400668621063
Projected weights: [0.24305214 0.19343325 0.10423806 0.45927653]
Weight sum: 1.0
True label: 0
Prediction: 0

[OK] PPO raw action → projected weights → ensemble → reward works.


In [42]:
smoke_agent_config = PPOConfig(
    rollout_steps=256,
    batch_size=64,
    n_epochs=2,
    lr=3e-4,
    device=DEVICE,
)

smoke_agent = PPOAgent(smoke_agent_config)

print("Smoke PPO agent created.")
print("Device:", smoke_agent.device)
print("Rollout steps:", smoke_agent_config.rollout_steps)

Smoke PPO agent created.
Device: cuda
Rollout steps: 256


In [44]:
smoke_dir = Path(tempfile.mkdtemp(prefix="ppo_smoke_"))

smoke_checkpoint_dir = smoke_dir / "checkpoints"
smoke_log_path = smoke_dir / "train_log.jsonl"

smoke_config = TrainerConfig(
    total_timesteps=512,          # 2 PPO updates
    eval_every_updates=1,         # required by Trainer
    eval_episodes=1,              # required by Trainer
    checkpoint_dir=str(smoke_checkpoint_dir),
    checkpoint_every_updates=2,
    log_path=str(smoke_log_path),
    verbose=True,
    seed=SEED,
)

smoke_trainer = Trainer(
    smoke_agent,
    train_env,
    val_env,
    smoke_config,
)

print("Smoke-test trainer created.")
print("Output directory:", smoke_dir)

Smoke-test trainer created.
Output directory: /tmp/ppo_smoke_q2973d5h


In [45]:
smoke_history = smoke_trainer.train()

print("\nSmoke training completed.")
print("Updates:", len(smoke_history))
print("Timesteps:", smoke_trainer._n_timesteps)

[update    1] timesteps=    256 train_mean_reward=+0.2578 policy_loss=-0.0086 value_loss=10.7015 | val_mean_reward=+0.2367 val_f1=0.4131
[update    2] timesteps=    512 train_mean_reward=+0.3203 policy_loss=-0.0061 value_loss=14.8251 | val_mean_reward=+0.2394 val_f1=0.4141

Smoke training completed.
Updates: 2
Timesteps: 512


In [46]:
assert len(smoke_history) == 2
assert smoke_trainer._n_updates == 2
assert smoke_trainer._n_timesteps == 512

for i, record in enumerate(smoke_history, start=1):
    print(f"\nUpdate {i}")
    print("Train mean reward:", record.get("train_mean_reward"))
    print("Policy loss:", record.get("policy_loss"))
    print("Value loss:", record.get("value_loss"))
    print("Entropy:", record.get("entropy"))

print("\n[OK] PPO smoke training completed successfully.")


Update 1
Train mean reward: 0.2578125
Policy loss: -0.00856980332173407
Value loss: 10.70146358013153
Entropy: 5.678253352642059

Update 2
Train mean reward: 0.3203125
Policy loss: -0.0060897283256053925
Value loss: 14.825056433677673
Entropy: 5.680968701839447

[OK] PPO smoke training completed successfully.


In [47]:
checkpoint_path = smoke_checkpoint_dir / "update_2.pt"

print("Checkpoint:", checkpoint_path)
print("Exists:", checkpoint_path.exists())

assert checkpoint_path.exists()

print("[OK] PPO checkpoint created.")

Checkpoint: /tmp/ppo_smoke_q2973d5h/checkpoints/update_2.pt
Exists: True
[OK] PPO checkpoint created.
